<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Python for Finance (2nd ed.)

**Mastering Data-Driven Finance**

&copy; Dr. Yves J. Hilpisch | The Python Quants GmbH

<img src="https://hilpisch.com/images/py4fi_2nd_shadow.png" width="300px" align="left">

# Financial Time Series

In [ ]:
import numpy as np
import pandas as pd
from pylab import mpl, plt
plt.style.use('seaborn-v0_8')
mpl.rcParams['font.family'] = 'serif'
%config InlineBackend.figure_format = 'svg'

In [ ]:
import warnings
warnings.simplefilter('ignore')

## Financial Data

### Data Import

In [ ]:
filename = '../../source/tr_eikon_eod_data.csv'  

In [ ]:
f = open(filename, 'r')  
f.readlines()[:5]  

In [ ]:
data = pd.read_csv(filename,  
                   index_col=0, 
                   parse_dates=True)  

In [ ]:
data.info()  

In [ ]:
data.head()  

In [ ]:
data.tail()  

In [ ]:
data.plot(figsize=(10, 12), subplots=True);  

In [ ]:
instruments = ['Apple Stock', 'Microsoft Stock',
               'Intel Stock', 'Amazon Stock', 'Goldman Sachs Stock',
               'SPDR S&P 500 ETF Trust', 'S&P 500 Index',
               'VIX Volatility Index', 'EUR/USD Exchange Rate',
               'Gold Price', 'VanEck Vectors Gold Miners ETF',
               'SPDR Gold Trust']

In [ ]:
for ric, name in zip(data.columns, instruments):
    print('{:8s} | {}'.format(ric, name))

### Summary Statistics

In [ ]:
data.info()  

In [ ]:
data.describe().round(2)  

In [ ]:
data.mean()  

In [ ]:
data.aggregate([min,  
                np.mean,  
                np.std,  
                np.median,  
                max]  
).round(2)

### Changes Over Time

In [ ]:
data.diff().head()  

In [ ]:
data.diff().mean()  

## Percentage changes
From a statistics point of view, absolute changes are not optimal because they
are dependent on the scale of the time series data itself. Therefore, percentage
changes are usually preferred. The following code derives the percentage changes
or percentage returns (also: simple returns) in a financial context and
visualizes their mean values per column.

In [ ]:
data.pct_change().round(3).head()  

In [ ]:
data.pct_change().mean().plot(kind='bar', figsize=(10, 6));  

### Logarithmic change
As an alternative to percentage returns, log returns can be used. In some
scenarios, they are easier to handle and therefore often preferred in a financial
context. Figure shows the cumulative log returns for the single financial
time series. This type of plot leads to some form of normalization:

In [ ]:
rets = np.log(data / data.shift(1))  

In [ ]:
rets.head().round(3)  

In [ ]:
rets.cumsum().apply(np.exp).plot(figsize=(10, 6));  

### Resampling
Resampling is an important operation on financial time series data. Usually this
takes the form of downsampling, meaning that, for example, a tick data series is
resampled to one-minute intervals or a time series with daily observations is
resampled to one with weekly or monthly observations.

### AVOIDING FORESIGHT BIAS
When resampling, pandas takes by default in many cases the left label (or index value) of the
interval. To be financially consistent, make sure to use the right label (index value) and in
general the last available data point in the interval. Otherwise, a foresight bias might sneak into
the financial analysis.3

In [ ]:
data.resample('1w', label='right').last().head()  

In [ ]:
data.resample('1m', label='right').last().head()  

This plots the cumulative log returns over time: first, the cumsum() method
is called, then np.exp() is applied to the results; finally, the resampling
takes place.

In [ ]:
rets.cumsum().apply(np.exp). resample('1m', label='right').last(
                          ).plot(figsize=(10, 6));

## Rolling Statistics

In [ ]:
sym = 'AAPL.O'

In [ ]:
data = pd.DataFrame(data[sym]).dropna()

In [ ]:
data.tail()

### An Overview

In [ ]:
window = 20  

In [ ]:
data['min'] = data[sym].rolling(window=window).min()  

In [ ]:
data['mean'] = data[sym].rolling(window=window).mean()  

In [ ]:
data['std'] = data[sym].rolling(window=window).std()  

In [ ]:
data['median'] = data[sym].rolling(window=window).median()  

In [ ]:
data['max'] = data[sym].rolling(window=window).max()  

In [ ]:
data['ewma'] = data[sym].ewm(halflife=0.5, min_periods=window).mean()  

In [ ]:
data.dropna().head()

In [ ]:
ax = data[['min', 'mean', 'max']].iloc[-200:].plot(
    figsize=(10, 6), style=['g--', 'r--', 'g--'], lw=0.8)  
data[sym].iloc[-200:].plot(ax=ax, lw=2.0);  

### A Technical Analysis Example

In [ ]:
data['SMA1'] = data[sym].rolling(window=42).mean()  

In [ ]:
data['SMA2'] = data[sym].rolling(window=252).mean()  

In [ ]:
data[[sym, 'SMA1', 'SMA2']].tail()

In [ ]:
data[[sym, 'SMA1', 'SMA2']].plot(figsize=(10, 6));  

In [ ]:
data.dropna(inplace=True)  

In [ ]:
data['positions'] = np.where(data['SMA1'] > data['SMA2'],  
                             1,  
                             -1)  

In [ ]:
ax = data[[sym, 'SMA1', 'SMA2', 'positions']].plot(figsize=(10, 6),
                                              secondary_y='positions')
ax.get_legend().set_bbox_to_anchor((0.25, 0.85));

## Regression Analysis

### The Data

In [ ]:
# EOD data from Thomson Reuters Eikon Data API
raw = pd.read_csv('../../source/tr_eikon_eod_data.csv',
                 index_col=0, parse_dates=True)

In [ ]:
data = raw[['.SPX', '.VIX']].dropna()

In [ ]:
data.tail()

In [ ]:
data.plot(subplots=True, figsize=(10, 6));

In [ ]:
data.loc[:'2012-12-31'].plot(secondary_y='.VIX', figsize=(10, 6));  

### Log Returns

In [ ]:
rets = np.log(data / data.shift(1)) 

In [ ]:
rets.head()

In [ ]:
rets.dropna(inplace=True)

In [ ]:
rets.plot(subplots=True, figsize=(10, 6));

In [ ]:
pd.plotting.scatter_matrix(rets,  
                           alpha=0.2,  
                           diagonal='hist',  
                           hist_kwds={'bins': 35},  
                           figsize=(10, 6));

### OLS Regression

In [ ]:
reg = np.polyfit(rets['.SPX'], rets['.VIX'], deg=1)  

In [ ]:
ax = rets.plot(kind='scatter', x='.SPX', y='.VIX', figsize=(10, 6))  
ax.plot(rets['.SPX'], np.polyval(reg, rets['.SPX']), 'r', lw=2);  

### Correlation

In [ ]:
rets.corr()  

In [ ]:
ax = rets['.SPX'].rolling(window=252).corr(
                  rets['.VIX']).plot(figsize=(10, 6))  
ax.axhline(rets.corr().iloc[0, 1], c='r');  

## High Frequency Data

In [ ]:
# from fxcmpy import fxcmpy_tick_data_reader as tdr
# data = tdr('EURUSD', start='2018-6-25', end='2018-06-30')
# data.get_data(start='2018-6-29',
#               end='2018-06-30').to_csv('../../source/fxcm_eur_usd_tick_data.csv')

In [ ]:
%%time
# data from FXCM Forex Capital Markets Ltd.
tick = pd.read_csv('../../source/fxcm_eur_usd_tick_data.csv',
                     index_col=0, parse_dates=True)

In [ ]:
tick.info()

In [ ]:
tick['Mid'] = tick.mean(axis=1)  

In [ ]:
tick['Mid'].plot(figsize=(10, 6));

In [ ]:
tick_resam = tick.resample(rule='5min', label='right').last()

In [ ]:
tick_resam.head()

In [ ]:
tick_resam['Mid'].plot(figsize=(10, 6));

<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="https://tpq.io" target="_blank">https://tpq.io</a> | <a href="https://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>